In [11]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


In [12]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.14.1+cpu
CUDA available: False


In [19]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Load data
df = pd.read_csv(
    "../data/raw/PS_20174392719_1491204439457_log.csv"
)

# Feature engineering
df["orig_balance_change"] = (
    df["oldbalanceOrg"] - df["newbalanceOrig"]
)

df["dest_balance_change"] = (
    df["newbalanceDest"] - df["oldbalanceDest"]
)

df["orig_amount_error"] = (
    df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"]
).abs()

df["dest_amount_error"] = (
    df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]
).abs()

# Select features and target
features = [
    "step", "type", "amount",
    "oldbalanceOrg", "newbalanceOrig",
    "oldbalanceDest", "newbalanceDest",
    "orig_balance_change", "dest_balance_change",
    "orig_amount_error", "dest_amount_error"
]

X = df[features]
y = df["isFraud"]

# Split: keep test set untouched
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (5090096, 11)
Testing: (1272524, 11)


In [20]:
X_sample, _, y_sample, _ = train_test_split(
    X_train,
    y_train,
    train_size=500_000,
    random_state=42,
    stratify=y_train
)

categorical = ["type"]

numeric = [c for c in features if c != "type"]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("num", "passthrough", numeric)
])

X_sample_processed = preprocessor.fit_transform(X_sample)
X_test_processed = preprocessor.transform(X_test)

scaler = StandardScaler()

X_sample_scaled = scaler.fit_transform(X_sample_processed)
X_test_scaled = scaler.transform(X_test_processed)

print("Sample ready:", X_sample_scaled.shape)

Sample ready: (500000, 15)


In [21]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

# Use float32 to reduce memory usage
X_np = np.asarray(X_sample_scaled, dtype=np.float32)
y_np = np.asarray(y_sample, dtype=np.float32)

# Split sample into training and validation sets
X_train_np, X_val_np, y_train_np, y_val_np = train_test_split(
    X_np,
    y_np,
    test_size=0.2,
    random_state=42,
    stratify=y_np
)

X_train_dl = torch.tensor(X_train_np, dtype=torch.float32)
y_train_dl = torch.tensor(y_train_np.reshape(-1, 1), dtype=torch.float32)
X_val_dl = torch.tensor(X_val_np, dtype=torch.float32)
y_val_dl = torch.tensor(y_val_np.reshape(-1, 1), dtype=torch.float32)

train_loader = DataLoader(
    TensorDataset(X_train_dl, y_train_dl),
    batch_size=2048,
    shuffle=True
)

print("Training rows:", len(X_train_dl))
print("Validation rows:", len(X_val_dl))
print("Fraud cases in training:", int(y_train_np.sum()))
print("Fraud cases in validation:", int(y_val_np.sum()))

Training rows: 400000
Validation rows: 100000
Fraud cases in training: 516
Fraud cases in validation: 129


In [22]:
class FraudNeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)


model = FraudNeuralNetwork(input_size=X_train_dl.shape[1])

# Give fraudulent transactions more weight during training
fraud_count = y_train_np.sum()
normal_count = len(y_train_np) - fraud_count

pos_weight = torch.tensor(
    [normal_count / fraud_count],
    dtype=torch.float32
)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(model)
print("Fraud weight:", round(pos_weight.item(), 2))

FraudNeuralNetwork(
  (network): Sequential(
    (0): Linear(in_features=15, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=32, out_features=1, bias=True)
  )
)
Fraud weight: 774.19


In [23]:
epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()

        logits = model(batch_X)
        loss = criterion(logits, batch_y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(batch_X)

    average_loss = total_loss / len(train_loader.dataset)

    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"Loss: {average_loss:.4f}"
    )

Epoch 1/5 | Loss: 0.6745
Epoch 2/5 | Loss: 0.3594
Epoch 3/5 | Loss: 0.3027
Epoch 4/5 | Loss: 0.2934
Epoch 5/5 | Loss: 0.2677


In [24]:
model.eval()

with torch.no_grad():
    logits = model(X_val_dl)
    probabilities = torch.sigmoid(logits).numpy().ravel()

# Initial threshold; we can tune it later
predictions = (probabilities >= 0.5).astype(int)

print("Precision:", round(
    precision_score(y_val_np, predictions, zero_division=0), 4
))
print("Recall:", round(
    recall_score(y_val_np, predictions, zero_division=0), 4
))
print("F1 Score:", round(
    f1_score(y_val_np, predictions, zero_division=0), 4
))
print("PR-AUC:", round(
    average_precision_score(y_val_np, probabilities), 4
))
print("Confusion Matrix:")
print(confusion_matrix(y_val_np, predictions))

Precision: 0.0143
Recall: 0.9767
F1 Score: 0.0282
PR-AUC: 0.5827
Confusion Matrix:
[[91178  8693]
 [    3   126]]


In [25]:
thresholds = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99]

print(f"{'Threshold':<12}{'Precision':<12}{'Recall':<12}{'F1':<12}{'Flagged':<10}")

for threshold in thresholds:
    predictions = (probabilities >= threshold).astype(int)

    precision = precision_score(
        y_val_np, predictions, zero_division=0
    )
    recall = recall_score(
        y_val_np, predictions, zero_division=0
    )
    f1 = f1_score(
        y_val_np, predictions, zero_division=0
    )
    flagged = int(predictions.sum())

    print(
        f"{threshold:<12.2f}{precision:<12.4f}"
        f"{recall:<12.4f}{f1:<12.4f}{flagged:<10}"
    )

Threshold   Precision   Recall      F1          Flagged   
0.50        0.0143      0.9767      0.0282      8819      
0.60        0.0182      0.9767      0.0357      6925      
0.70        0.0236      0.9690      0.0462      5287      
0.80        0.0322      0.9380      0.0623      3755      
0.90        0.0554      0.9070      0.1045      2111      
0.95        0.1007      0.8295      0.1795      1063      
0.99        0.3821      0.6279      0.4751      212       
